# WavqWise: Trading Forecast with REAL Stock Data
**Sense. Forecast. Alert.**

Real AAPL/TSLA/MSFT data from Yahoo Finance via yfinance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VK-Ant/wavqwise/blob/main/demos/notebooks/wavqwise_trading_real_data.ipynb)

**Author:** [VK-Ant](https://github.com/VK-Ant)

In [ ]:
!pip install wavqwise yfinance -q

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from wavqwise import WavqPipeline
from wavqwise.trading.indicators.momentum import RSIIndicator, StochasticIndicator
from wavqwise.trading.indicators.trend import MACDIndicator, SMAIndicator
from wavqwise.trading.indicators.volatility import BollingerBandsIndicator, ATRIndicator
print('Ready')

## 1. Download Real Stock Data

In [ ]:
TICKER = 'AAPL'  # Change to any ticker: TSLA, MSFT, GOOGL, AMZN
stock = yf.download(TICKER, period='2y', auto_adjust=True, progress=False).reset_index()
if isinstance(stock.columns, pd.MultiIndex):
    stock.columns = [c[0] if c[1]=='' else c[0] for c in stock.columns]
print(f'{TICKER}: {stock["Date"].iloc[0].date()} to {stock["Date"].iloc[-1].date()}')
print(f'Latest: ${stock["Close"].iloc[-1]:.2f}')
print(f'Return: {(stock["Close"].iloc[-1]/stock["Close"].iloc[0]-1)*100:+.1f}%')
stock.tail()

## 2. Technical Indicators

In [ ]:
stock = RSIIndicator(14).compute(stock)
stock = MACDIndicator().compute(stock)
stock = SMAIndicator(20).compute(stock)
stock = SMAIndicator(50).compute(stock)
stock = SMAIndicator(200).compute(stock)
stock = BollingerBandsIndicator(20, 2).compute(stock)
stock = ATRIndicator(14).compute(stock)
stock = StochasticIndicator(14).compute(stock)

l = stock.dropna().iloc[-1]
print(f'RSI: {l["RSI"]:.1f} | MACD: {l["MACD"]:.4f}')
print(f'SMA-20: ${l["SMA_20"]:.2f} | SMA-50: ${l["SMA_50"]:.2f} | SMA-200: ${l["SMA_200"]:.2f}')
print(f'Trend: {"Bullish" if l["SMA_20"]>l["SMA_50"]>l["SMA_200"] else "Bearish" if l["SMA_20"]<l["SMA_50"]<l["SMA_200"] else "Mixed"}')

## 3. Forecast with WavqPipeline

In [ ]:
pipeline = WavqPipeline()
pipeline.load(stock, target='Close', time='Date')

for model in ['moving_average', 'ema', 'naive']:
    fc = pipeline.forecast(horizon=30, model=model)
    print(f'{model}: day1=${fc.forecast["Close"].iloc[0]:.2f}, day30=${fc.forecast["Close"].iloc[-1]:.2f}')

In [ ]:
# Model comparison
comp = pipeline.compare_models(['moving_average','ema','naive','seasonal_naive'], horizon=14)
print(comp.to_string(index=False))
print(f'Best: {comp.iloc[0]["model"]}')

## 4. Full Trading Chart

In [ ]:
best_model = comp.iloc[0]['model']
forecast = pipeline.forecast(horizon=30, model=best_model)
clean = stock.dropna()

fig, axes = plt.subplots(4, 1, figsize=(16, 18), gridspec_kw={'height_ratios': [4, 1, 1, 1]})
fig.suptitle(f'WavqWise: {TICKER} Real-Time Analysis', fontsize=16, fontweight='bold')

ax = axes[0]
ax.plot(clean['Date'], clean['Close'], color='#1e293b', linewidth=1.2, label='Close')
ax.plot(clean['Date'], clean['SMA_20'], color='#2563eb', linewidth=0.7, alpha=0.6, label='SMA-20')
ax.plot(clean['Date'], clean['SMA_50'], color='#dc2626', linewidth=0.7, alpha=0.6, label='SMA-50')
ax.fill_between(clean['Date'], clean['BB_lower'], clean['BB_upper'], alpha=0.08, color='#6366f1')
fc = forecast.forecast
ax.plot(fc['Date'], fc['Close'], '--', color='#f59e0b', linewidth=2.5, label=f'Forecast ({best_model})')
ax.fill_between(fc['Date'], fc['Close_lower'], fc['Close_upper'], alpha=0.15, color='#f59e0b')
ax.legend(fontsize=8, ncol=2); ax.set_ylabel('Price ($)'); ax.grid(True, alpha=0.3)

axes[1].bar(clean['Date'], clean['Volume'], color=['#059669' if c>=o else '#dc2626' for c,o in zip(clean['Close'],clean['Open'])], alpha=0.5, width=1)
axes[1].set_ylabel('Volume'); axes[1].grid(True, alpha=0.3)

axes[2].plot(clean['Date'], clean['RSI'], color='#7c3aed')
axes[2].axhline(70, color='#dc2626', ls='--', alpha=0.5); axes[2].axhline(30, color='#059669', ls='--', alpha=0.5)
axes[2].set_ylabel('RSI'); axes[2].set_ylim(0,100); axes[2].grid(True, alpha=0.3)

axes[3].plot(clean['Date'], clean['MACD'], color='#2563eb', label='MACD')
axes[3].plot(clean['Date'], clean['MACD_signal'], color='#dc2626', label='Signal')
axes[3].bar(clean['Date'], clean['MACD_hist'], color=['#059669' if v>=0 else '#dc2626' for v in clean['MACD_hist']], alpha=0.5, width=1)
axes[3].legend(fontsize=8); axes[3].set_ylabel('MACD'); axes[3].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 5. Multi-Stock Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for ticker, color in [('AAPL','#2563eb'),('TSLA','#dc2626'),('MSFT','#059669')]:
    df = yf.download(ticker, period='1y', auto_adjust=True, progress=False)
    normalized = df['Close'] / df['Close'].iloc[0] * 100
    ax.plot(normalized.index, normalized, color=color, linewidth=1.5, label=ticker)
ax.set_title('Normalized Price Comparison (1Y)')
ax.set_ylabel('Normalized (base=100)'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
**WavqWise** - Sense. Forecast. Alert. | [GitHub](https://github.com/VK-Ant/wavqwise) | [PyPI](https://pypi.org/project/wavqwise/)